# BOS Breakout V1 Validation (Grid Search + Time-Split + Walk-Forward)

This notebook consolidates **grid search**, **time-split validation**, and **walk-forward validation** into one consistent research workflow.

Deterministic V1 rules used everywhere:
- Signal is detected on **Close[t]**.
- Entry is executed on **Open[t+1]**.
- Exit uses OHLC-only rules with a deterministic tie-breaker for same-bar SL/TP.

Code style: PEP8-ish formatting, readable functions, and English docstrings.


In [ ]:
NOTEBOOK_VERSION = "bos_v1_validation_merged_v5"
print("Running", NOTEBOOK_VERSION)

Running bos_v1_validation_merged_v5


In [2]:
# region imports
from AlgorithmImports import *
# endregion

from __future__ import annotations

import math
from dataclasses import dataclass
from datetime import datetime, timedelta
from typing import Any, Dict, Iterable, List, Optional, Tuple

import numpy as np
import pandas as pd

# Project modules (workspace files)
from entry_exit import (
    Bar,
    BosSignal,
    ExitReason,
    PositionDirection,
    SameBarSlTpRule,
    SwingLevels,
    TakeProfitMode,
    TradeExit,
    TradePlan,
    check_exit_rules,
    detect_bos_signal,
    plan_trade_from_signal,
    update_last_swing_levels,
)
from risk import RiskConfig
from stop_loss import StopLossManager
from swing_high_low_detection import swing_highs_lows_online


## 1) Data loading

This notebook is intended for QuantConnect Research (QuantBook). If you run it elsewhere, just provide a `df` with OHLC columns and a DateTime index.


In [3]:
def load_ohlc_from_quantbook(
    *,
    ticker: str,
    resolution: Resolution,
    start: datetime,
    end: datetime,
) -> pd.DataFrame:
    """Load OHLC history via QuantBook.

    Args:
        ticker: Symbol ticker, e.g. "BTCUSD" or "ETHUSD".
        resolution: QuantConnect resolution.
        start: Inclusive start datetime.
        end: Exclusive end datetime.

    Returns:
        DataFrame with columns: open, high, low, close and DateTimeIndex.

    Raises:
        RuntimeError: If QuantBook is unavailable or history is empty.
    """
    try:
        from QuantConnect.Research import QuantBook
    except BaseException as exc:
        raise RuntimeError("QuantBook is unavailable in this environment.") from exc

    qb = QuantBook()
    symbol = qb.AddCrypto(ticker, resolution).Symbol
    history = qb.History(symbol, start, end, resolution)
    if history is None or len(history) == 0:
        raise RuntimeError("History is empty. Check symbol, dates, and resolution.")

    df = history.loc[symbol].copy()
    df = df[["open", "high", "low", "close"]].dropna()
    df.index = pd.to_datetime(df.index)
    return df


# --- default dataset (edit if needed) ---
TICKER = "ETHUSD"
RESOLUTION = Resolution.Hour
START = datetime(2023, 1, 1)
END = datetime(2025, 1, 1)

# Running inside QC Research? Then this works.
# Otherwise, set df manually.
df = load_ohlc_from_quantbook(ticker=TICKER, resolution=RESOLUTION, start=START, end=END)

df.head()


,open,high,low,close
time,,,,
2023-01-01 06:00:00,1192.15,1193.83,1191.76,1193.44
2023-01-01 07:00:00,1193.44,1194.42,1192.96,1193.87
2023-01-01 08:00:00,1193.88,1194.95,1192.07,1194.27
2023-01-01 09:00:00,1194.22,1195.27,1193.21,1193.49
2023-01-01 10:00:00,1193.49,1195.94,1193.02,1195.59


## 2) Swings (non-repainting)

We generate confirmed swing points first, then run the deterministic BOS logic using these levels.


In [4]:
SWING_PARAMS = {
    "N_candidates": [5, 10, 20],
    "N_confirmation": 3,
    "min_move_threshold": 0.0,
    "min_bars_between_swings": 3,
}

swings = swing_highs_lows_online(
    df[["close", "high", "low"]].copy(),
    **SWING_PARAMS,
)

swings.dropna().head()


,HighLow,Level
time,,
2023-01-01 06:00:00,1.0,1193.83
2023-01-01 10:00:00,1.0,1195.94
2023-01-01 13:00:00,1.0,1198.11
2023-01-01 17:00:00,1.0,1197.70
2023-01-01 20:00:00,1.0,1203.86


## 3) Strategy configuration space

We build a configuration list, including the original grid and a few extra "my" configs for sanity.


In [5]:
@dataclass(frozen=True)
class StrategyConfig:
    """Parameter bundle for the BOS breakout V1 backtest."""

    sl_mode: str
    tp_mode: TakeProfitMode
    fixed_pct: float
    buffer_pct: float
    tp_mult: float
    same_bar_rule: SameBarSlTpRule
    cooldown_bars: int = 0
    max_trades_per_day: Optional[int] = None


def build_config_grid() -> List[StrategyConfig]:
    """Create a configuration grid.

    Returns:
        List of StrategyConfig.
    """
    sl_modes = ["fixed", "structural", "bos"]
    tp_modes = [TakeProfitMode.RR_BASED, TakeProfitMode.RANGE_BASED]

    fixed_grid = [0.0025, 0.005, 0.01]
    buffer_grid = [0.0, 0.001, 0.002, 0.003]
    tp_mult_grid = [1.0, 1.5, 2.0, 3.0]

    same_bar_rules = [SameBarSlTpRule.WORST_CASE, SameBarSlTpRule.OPEN_PROXIMITY]

    configs: List[StrategyConfig] = []
    for sl_mode in sl_modes:
        for tp_mode in tp_modes:
            for fixed_pct in fixed_grid:
                for buffer_pct in buffer_grid:
                    for tp_mult in tp_mult_grid:
                        for same_bar_rule in same_bar_rules:
                            # Only fixed uses fixed_pct.
                            if sl_mode != "fixed" and fixed_pct != fixed_grid[0]:
                                continue
                            # RANGE_BASED ignores tp_mult.
                            if tp_mode == TakeProfitMode.RANGE_BASED and tp_mult != tp_mult_grid[0]:
                                continue

                            configs.append(
                                StrategyConfig(
                                    sl_mode=sl_mode,
                                    tp_mode=tp_mode,
                                    fixed_pct=fixed_pct,
                                    buffer_pct=buffer_pct,
                                    tp_mult=tp_mult,
                                    same_bar_rule=same_bar_rule,
                                    cooldown_bars=0,
                                    max_trades_per_day=None,
                                )
                            )

    # A few hand-picked configs for quick comparison.
    configs += [
        StrategyConfig(
            sl_mode="structural",
            tp_mode=TakeProfitMode.RANGE_BASED,
            fixed_pct=fixed_grid[0],
            buffer_pct=0.0015,
            tp_mult=1.0,
            same_bar_rule=SameBarSlTpRule.OPEN_PROXIMITY,
        ),
        StrategyConfig(
            sl_mode="bos",
            tp_mode=TakeProfitMode.RR_BASED,
            fixed_pct=fixed_grid[0],
            buffer_pct=0.001,
            tp_mult=2.0,
            same_bar_rule=SameBarSlTpRule.WORST_CASE,
        ),
    ]

    return configs


configs = build_config_grid()
len(configs)


202

## 4) Backtest engine (deterministic V1)

This is a small research backtester. It does not place real QC orders; it simulates trade outcomes from OHLC.


In [6]:
def _bars_from_df(df_ohlc: pd.DataFrame) -> List[Bar]:
    """Convert OHLC DataFrame to a list of Bar objects."""
    out: List[Bar] = []
    for ts, row in df_ohlc.iterrows():
        out.append(
            Bar(
                open=float(row["open"]),
                high=float(row["high"]),
                low=float(row["low"]),
                close=float(row["close"]),
                time=str(ts),
            )
        )
    return out


def _compute_trade_metrics(trades: pd.DataFrame) -> Dict[str, Any]:
    """Compute simple performance metrics from a trades table."""
    if trades.empty:
        return {
            "num_trades": 0,
            "total_pnl": 0.0,
            "winrate": float("nan"),
            "avg_r": float("nan"),
            "profit_factor": float("nan"),
            "max_dd_pnl": 0.0,
        }

    pnl = trades["pnl"].to_numpy(dtype=float)
    wins = pnl[pnl > 0]
    losses = pnl[pnl < 0]

    total_pnl = float(np.sum(pnl))
    winrate = float(np.mean(pnl > 0))
    avg_r = float(np.mean(trades["r_mult"].to_numpy(dtype=float)))

    profit_factor = float("nan")
    if len(losses) > 0:
        denom = abs(float(np.sum(losses)))
        profit_factor = float(np.sum(wins) / denom) if denom > 0 else float("nan")

    # Drawdown over cumulative PnL (cash terms)
    cum = np.cumsum(pnl)
    peak = np.maximum.accumulate(cum)
    dd = cum - peak
    max_dd_pnl = float(np.min(dd)) if len(dd) else 0.0

    return {
        "num_trades": int(len(trades)),
        "total_pnl": total_pnl,
        "winrate": winrate,
        "avg_r": avg_r,
        "profit_factor": profit_factor,
        "max_dd_pnl": max_dd_pnl,
    }


def backtest_bos_breakout(
    df_ohlc: pd.DataFrame,
    swings: pd.DataFrame,
    config: StrategyConfig,
    risk: RiskConfig,
) -> Dict[str, Any]:
    """Run a deterministic V1 BOS breakout backtest.

    Args:
        df_ohlc: OHLC data (open/high/low/close) indexed by datetime.
        swings: Output of swing_highs_lows_online aligned to df_ohlc.
        config: Strategy configuration.
        risk: Risk configuration used for sizing.

    Returns:
        Dict with keys: trades (DataFrame) and metrics (dict).
    """
    df_ohlc = df_ohlc.dropna().copy()
    swings = swings.reindex(df_ohlc.index)

    bars = _bars_from_df(df_ohlc)

    stop_loss_manager = StopLossManager(
        mode=config.sl_mode,
        fixed_pct=config.fixed_pct,
        buffer_pct=config.buffer_pct,
    )

    swing_levels = SwingLevels()
    last_valid_range: Optional[Tuple[float, float]] = None

    state: str = "FLAT"  # FLAT/LONG/SHORT
    trade_plan: Optional[TradePlan] = None

    cooldown_until_bar: int = -1
    current_day: Optional[pd.Timestamp] = None
    trades_today: int = 0

    trades: List[Dict[str, Any]] = []

    for i in range(len(bars)):
        ts = df_ohlc.index[i]
        bar = bars[i]

        # Daily reset for max_trades_per_day
        day = pd.Timestamp(ts).normalize()
        if current_day is None or day != current_day:
            current_day = day
            trades_today = 0

        # 1) Update swings (confirmed swings only)
        hl = swings.iloc[i]["HighLow"] if "HighLow" in swings.columns else np.nan
        lvl = swings.iloc[i]["Level"] if "Level" in swings.columns else np.nan

        if pd.notna(hl) and pd.notna(lvl):
            swing_levels = update_last_swing_levels(
                swing_levels,
                highlow_flag=float(hl),
                level=float(lvl),
            )

            if config.tp_mode == TakeProfitMode.RANGE_BASED:
                hi = swing_levels.last_swing_high_price
                lo = swing_levels.last_swing_low_price
                if hi is not None and lo is not None and hi > lo:
                    last_valid_range = (float(hi), float(lo))

        # 2) Exit (start checking from the entry bar)
        if state in ("LONG", "SHORT") and trade_plan is not None:
            if i >= int(trade_plan.entry_candle_index):
                exit_event = check_exit_rules(
                    bar=bar,
                    direction=trade_plan.direction,
                    sl_price=float(trade_plan.sl_price),
                    tp_price=float(trade_plan.tp_price),
                    same_bar_rule=config.same_bar_rule,
                )

                if exit_event is not None:
                    entry_price = float(trade_plan.entry_price)
                    exit_price = float(exit_event.exit_price)
                    qty = float(trade_plan.quantity)

                    pnl = (
                        (exit_price - entry_price) * qty
                        if trade_plan.direction == PositionDirection.LONG
                        else (entry_price - exit_price) * qty
                    )

                    risk_per_unit = abs(entry_price - float(trade_plan.sl_price))
                    r_mult = float(pnl / (risk_per_unit * qty)) if risk_per_unit > 0 else float("nan")

                    trades.append(
                        {
                            "entry_time": bars[int(trade_plan.entry_candle_index)].time,
                            "exit_time": bar.time,
                            "direction": trade_plan.direction.value,
                            "entry_price": entry_price,
                            "sl_price": float(trade_plan.sl_price),
                            "tp_price": float(trade_plan.tp_price),
                            "exit_price": exit_price,
                            "exit_reason": exit_event.exit_reason.value,
                            "qty": qty,
                            "pnl": float(pnl),
                            "r_mult": r_mult,
                            "signal_index": int(trade_plan.signal_candle_index),
                            "entry_index": int(trade_plan.entry_candle_index),
                            "exit_index": int(i),
                        }
                    )

                    state = "FLAT"
                    trade_plan = None
                    stop_loss_manager.reset()

                    if config.cooldown_bars and config.cooldown_bars > 0:
                        cooldown_until_bar = i + int(config.cooldown_bars)

                    continue

        # 3) Entry
        if state != "FLAT":
            continue

        if config.max_trades_per_day is not None and trades_today >= int(config.max_trades_per_day):
            continue

        if i < cooldown_until_bar:
            continue

        # Evaluate BOS on bar i (signal), enter on bar i+1 open.
        bos_signal = detect_bos_signal(bars=bars, t=i, swing_levels=swing_levels)
        if bos_signal is None:
            continue

        # RANGE_BASED requires a valid stored range (avoid NaN issues).
        effective_swing_levels = swing_levels
        if config.tp_mode == TakeProfitMode.RANGE_BASED:
            if last_valid_range is None:
                continue
            hi, lo = last_valid_range
            effective_swing_levels = SwingLevels(last_swing_high_price=hi, last_swing_low_price=lo)

        stop_loss_manager.reset()
        try:
            plan = plan_trade_from_signal(
                bars=bars,
                bos_signal=bos_signal,
                swing_levels=effective_swing_levels,
                stop_loss_manager=stop_loss_manager,
                tp_mode=config.tp_mode,
                tp_mult=config.tp_mult,
                risk_config=risk,
                buying_power_cash=None,
            )
        except BaseException:
            stop_loss_manager.reset()
            continue

        trade_plan = plan
        state = "LONG" if plan.direction == PositionDirection.LONG else "SHORT"
        trades_today += 1

    trades_df = pd.DataFrame(trades)
    metrics = _compute_trade_metrics(trades_df)
    return {"trades": trades_df, "metrics": metrics}


## 5) Evaluation utilities

- `grid_search`: runs all configs on a dataset slice.
- `time_split`: optimizes on train and evaluates the chosen config on test.
- `walk_forward`: repeated optimization + OOS testing windows.


In [7]:
def grid_search(
    df_ohlc: pd.DataFrame,
    swings: pd.DataFrame,
    risk: RiskConfig,
    configs: List[StrategyConfig],
) -> pd.DataFrame:
    """Evaluate all configs on the provided dataset.

    This helper is intentionally robust for research: if a single configuration
    errors (e.g., impossible sizing), the row is recorded with an `error`
    message and the grid keeps running.
    """
    rows: List[Dict[str, Any]] = []
    for cfg in configs:
        err: Optional[str] = None
        metrics: Dict[str, Any] = {
            "num_trades": 0,
            "total_pnl": float("nan"),
            "winrate": float("nan"),
            "avg_r": float("nan"),
            "profit_factor": float("nan"),
            "max_dd_pnl": float("nan"),
        }

        try:
            res = backtest_bos_breakout(df_ohlc, swings, cfg, risk)
            metrics = res["metrics"]
        except BaseException as exc:
            err = f"{type(exc).__name__}: {exc}"

        rows.append(
            {
                "sl_mode": cfg.sl_mode,
                "tp_mode": cfg.tp_mode.value,
                "fixed_pct": cfg.fixed_pct,
                "buffer_pct": cfg.buffer_pct,
                "tp_mult": cfg.tp_mult,
                "same_bar_rule": cfg.same_bar_rule.value,
                "cooldown_bars": cfg.cooldown_bars,
                "max_trades_per_day": cfg.max_trades_per_day,
                "error": err,
                **metrics,
            }
        )

    return pd.DataFrame(rows)


def rank_results(df_results: pd.DataFrame) -> pd.DataFrame:
    """Rank results with a simple, readable scoring heuristic.

    The score is deliberately conservative:
    - total_pnl is good
    - max_dd_pnl is bad (more negative)
    - profit_factor helps break ties
    """
    out = df_results.copy()

    # Replace NaNs so sorting doesn't explode.
    out["profit_factor"] = out["profit_factor"].fillna(-np.inf)
    out["winrate"] = out["winrate"].fillna(0.0)

    # Score: reward pnl, penalize drawdown, nudge by profit_factor.
    out["score"] = out["total_pnl"] + 0.25 * out["max_dd_pnl"] + 10.0 * out["profit_factor"].clip(-10, 10)

    out = out.sort_values(["score", "num_trades"], ascending=[False, False]).reset_index(drop=True)
    return out


def select_best_config(
    ranked_results: pd.DataFrame,
) -> StrategyConfig:
    """Select best config from ranked results."""
    if ranked_results.empty:
        raise ValueError("No results to select from.")

    r = ranked_results.iloc[0]
    return StrategyConfig(
        sl_mode=str(r["sl_mode"]),
        tp_mode=TakeProfitMode(str(r["tp_mode"])),
        fixed_pct=float(r["fixed_pct"]),
        buffer_pct=float(r["buffer_pct"]),
        tp_mult=float(r["tp_mult"]),
        same_bar_rule=SameBarSlTpRule(str(r["same_bar_rule"])),
        cooldown_bars=int(r.get("cooldown_bars", 0) or 0),
        max_trades_per_day=None if pd.isna(r.get("max_trades_per_day", np.nan)) else int(r["max_trades_per_day"]),
    )


def time_split_evaluation(
    df_ohlc: pd.DataFrame,
    swings: pd.DataFrame,
    risk: RiskConfig,
    configs: List[StrategyConfig],
    split_frac: float = 0.7,
) -> Dict[str, Any]:
    """Optimize on train split and evaluate best config on test split."""
    if not 0.0 < split_frac < 1.0:
        raise ValueError("split_frac must be in (0, 1)")

    cut = int(len(df_ohlc) * split_frac)
    train_df, test_df = df_ohlc.iloc[:cut].copy(), df_ohlc.iloc[cut:].copy()
    train_sw, test_sw = swings.reindex(train_df.index), swings.reindex(test_df.index)

    train_results = rank_results(grid_search(train_df, train_sw, risk, configs))
    best_cfg = select_best_config(train_results)

    test_res = backtest_bos_breakout(test_df, test_sw, best_cfg, risk)

    return {
        "best_config": best_cfg,
        "train_table": train_results,
        "test_metrics": test_res["metrics"],
        "test_trades": test_res["trades"],
        "train_period": (train_df.index.min(), train_df.index.max()),
        "test_period": (test_df.index.min(), test_df.index.max()),
    }


def walk_forward_evaluation(
    df_ohlc: pd.DataFrame,
    swings: pd.DataFrame,
    risk: RiskConfig,
    configs: List[StrategyConfig],
    is_window_days: int = 180,
    oos_window_days: int = 60,
    anchored: bool = True,
) -> Dict[str, Any]:
    """Walk-forward: optimize on IS window, then evaluate on OOS window.

    Procedure per step:
    1. Define in-sample window.
    2. Grid search on IS.
    3. Select best config by rank_results.
    4. Backtest on the subsequent out-of-sample window.

    Args:
        is_window_days: In-sample window size in days.
        oos_window_days: Out-of-sample window size in days.
        anchored: If True, IS always starts from data start.

    Returns:
        Dict with aggregated OOS trades, per-window log, and overall metrics.
    """
    all_oos_trades: List[pd.DataFrame] = []
    log_rows: List[Dict[str, Any]] = []

    start_ts = pd.Timestamp(df_ohlc.index.min())
    end_ts = pd.Timestamp(df_ohlc.index.max())

    step_start = start_ts
    while True:
        train_end = step_start + timedelta(days=int(is_window_days))
        test_end = train_end + timedelta(days=int(oos_window_days))

        if test_end > end_ts:
            break

        train_start = start_ts if anchored else step_start

        is_df = df_ohlc.loc[train_start:train_end].copy()
        is_sw = swings.reindex(is_df.index)

        is_ranked = rank_results(grid_search(is_df, is_sw, risk, configs))
        if is_ranked.empty or int(is_ranked.iloc[0]["num_trades"]) == 0:
            step_start = step_start + timedelta(days=int(oos_window_days))
            continue

        best_cfg = select_best_config(is_ranked)

        oos_df = df_ohlc.loc[train_end:test_end].copy()
        oos_sw = swings.reindex(oos_df.index)
        oos_res = backtest_bos_breakout(oos_df, oos_sw, best_cfg, risk)

        oos_metrics = oos_res["metrics"]
        if not oos_res["trades"].empty:
            all_oos_trades.append(oos_res["trades"])

        log_rows.append(
            {
                "train_period": f"{train_start.date()} to {train_end.date()}",
                "test_period": f"{train_end.date()} to {test_end.date()}",
                "best_sl": best_cfg.sl_mode,
                "best_tp": best_cfg.tp_mode.value,
                "best_same_bar": best_cfg.same_bar_rule.value,
                "is_score": float(is_ranked.iloc[0]["score"]),
                "oos_total_pnl": float(oos_metrics["total_pnl"]),
                "oos_num_trades": int(oos_metrics["num_trades"]),
                "oos_profit_factor": float(oos_metrics["profit_factor"]) if not pd.isna(oos_metrics["profit_factor"]) else np.nan,
            }
        )

        step_start = step_start + timedelta(days=int(oos_window_days))

    oos_trades = pd.concat(all_oos_trades, ignore_index=True) if all_oos_trades else pd.DataFrame()
    oos_metrics = _compute_trade_metrics(oos_trades)

    return {
        "oos_trades": oos_trades,
        "oos_metrics": oos_metrics,
        "log": pd.DataFrame(log_rows),
        "anchored": anchored,
        "is_window_days": is_window_days,
        "oos_window_days": oos_window_days,
    }


## 6) Run everything

We evaluate:
- Full-sample grid search ranking
- Time-split validation
- Walk-forward validation (anchored + rolling)

At the end we print the "winner" based on out-of-sample performance.


In [8]:
risk = RiskConfig(
    risk_budget_cash=100.0,
    max_quantity=None,
    min_risk_per_unit=None,
    use_buying_power_cap=False,
)

# --- A) Full sample ranking ---
full_ranked = rank_results(grid_search(df, swings, risk, configs))
display(full_ranked.head(20))


,sl_mode,tp_mode,fixed_pct,buffer_pct,tp_mult,same_bar_rule,cooldown_bars,max_trades_per_day,error,num_trades,total_pnl,winrate,avg_r,profit_factor,max_dd_pnl,score
0,fixed,RR_BASED,0.0100,0.000,3.0,WORST_CASE,0,None,None,942,4125.61280,0.263270,0.053079,1.068540,-2216.917800,3582.068750
1,fixed,RR_BASED,0.0100,0.000,3.0,OPEN_PROXIMITY,0,None,None,942,4125.61280,0.263270,0.053079,1.068540,-2216.917800,3582.068750
2,fixed,RR_BASED,0.0100,0.001,3.0,WORST_CASE,0,None,None,942,4125.61280,0.263270,0.053079,1.068540,-2216.917800,3582.068750
3,fixed,RR_BASED,0.0100,0.001,3.0,OPEN_PROXIMITY,0,None,None,942,4125.61280,0.263270,0.053079,1.068540,-2216.917800,3582.068750
4,fixed,RR_BASED,0.0100,0.002,3.0,WORST_CASE,0,None,None,942,4125.61280,0.263270,0.053079,1.068540,-2216.917800,3582.068750
5,fixed,RR_BASED,0.0100,0.002,3.0,OPEN_PROXIMITY,0,None,None,942,4125.61280,0.263270,0.053079,1.068540,-2216.917800,3582.068750
6,fixed,RR_BASED,0.0100,0.003,3.0,WORST_CASE,0,None,None,942,4125.61280,0.263270,0.053079,1.068540,-2216.917800,3582.068750
7,fixed,RR_BASED,0.0100,0.003,3.0,OPEN_PROXIMITY,0,None,None,942,4125.61280,0.263270,0.053079,1.068540,-2216.917800,3582.068750
8,bos,RR_BASED,0.0025,0.003,3.0,OPEN_PROXIMITY,0,None,None,825,3882.95024,0.261818,0.047273,1.073704,-3816.132940,2939.654044
9,bos,RR_BASED,0.0025,0.003,3.0,WORST_CASE,0,None,None,825,3502.61392,0.260606,0.042424,1.066365,-3816.132940,2559.244333


In [9]:
# --- B) Time-split validation ---
time_split = time_split_evaluation(df, swings, risk, configs, split_frac=0.7)

best_time_cfg = time_split["best_config"]
print("TIME-SPLIT BEST CONFIG:", best_time_cfg)
print("TIME-SPLIT OOS METRICS:", time_split["test_metrics"])

display(time_split["test_trades"].head(10))


TIME-SPLIT BEST CONFIG: StrategyConfig(sl_mode='structural', tp_mode=<TakeProfitMode.RR_BASED: 'RR_BASED'>, fixed_pct=0.0025, buffer_pct=0.002, tp_mult=3.0, same_bar_rule=<SameBarSlTpRule.WORST_CASE: 'WORST_CASE'>, cooldown_bars=0, max_trades_per_day=None)
TIME-SPLIT OOS METRICS: {'num_trades': 112, 'total_pnl': -2806.054639999997, 'winrate': 0.17857142857142858, 'avg_r': -0.2857142857142857, 'profit_factor': 0.6265119466319553, 'max_dd_pnl': -3186.715899999996}


,entry_time,exit_time,direction,entry_price,sl_price,tp_price,exit_price,exit_reason,qty,pnl,r_mult,signal_index,entry_index,exit_index
0,2024-05-27 04:00:00,2024-05-28 04:00:00,LONG,3911.68,3825.85664,4169.15008,3825.85664,SL,1.0,-85.82336,-1.0,5,6,30
1,2024-05-28 11:00:00,2024-05-28 19:00:00,LONG,3877.20,3808.73560,4082.59320,3808.73560,SL,1.0,-68.46440,-1.0,36,37,45
2,2024-05-29 00:00:00,2024-05-29 09:00:00,LONG,3858.94,3807.95212,4011.90364,3807.95212,SL,1.0,-50.98788,-1.0,49,50,59
3,2024-05-29 11:00:00,2024-06-11 07:00:00,SHORT,3802.00,3891.84400,3532.46800,3532.46800,TP,1.0,269.53200,3.0,60,61,369
4,2024-06-11 23:00:00,2024-06-11 23:00:00,SHORT,3501.08,3509.19216,3476.74352,3509.19216,SL,12.0,-97.34592,-1.0,384,385,385
5,2024-06-12 01:00:00,2024-06-12 02:00:00,SHORT,3497.27,3509.18454,3461.52638,3461.52638,TP,8.0,285.94896,3.0,386,387,388
6,2024-06-12 05:00:00,2024-06-13 16:00:00,LONG,3509.07,3454.50186,3672.77442,3454.50186,SL,1.0,-54.56814,-1.0,390,391,426
7,2024-06-13 18:00:00,2024-06-15 07:00:00,SHORT,3469.51,3534.47902,3274.60294,3534.47902,SL,1.0,-64.96902,-1.0,427,428,465
8,2024-06-17 20:00:00,2024-06-18 01:00:00,LONG,3558.82,3475.47236,3808.86292,3475.47236,SL,1.0,-83.34764,-1.0,525,526,531
9,2024-06-18 10:00:00,2024-06-18 11:00:00,SHORT,3448.08,3458.41616,3417.07152,3417.07152,TP,9.0,279.07632,3.0,539,540,541


In [10]:
# --- C) Walk-forward: anchored ---
wf_anchored = walk_forward_evaluation(
    df, swings, risk, configs,
    is_window_days=180,
    oos_window_days=60,
    anchored=True,
)

print("WF-ANCHORED OOS METRICS:", wf_anchored["oos_metrics"])
display(wf_anchored["log"].head(20))


WF-ANCHORED OOS METRICS: {'num_trades': 989, 'total_pnl': -9704.585629999718, 'winrate': 0.2942366026289181, 'avg_r': -0.11217966597580739, 'profit_factor': 0.8461233664973696, 'max_dd_pnl': -10837.694149999741}


,train_period,test_period,best_sl,best_tp,best_same_bar,is_score,oos_total_pnl,oos_num_trades,oos_profit_factor
0,2023-01-01 to 2023-06-30,2023-06-30 to 2023-08-29,fixed,RANGE_BASED,OPEN_PROXIMITY,4740.388077,-616.99150,108,0.899746
1,2023-01-01 to 2023-08-29,2023-08-29 to 2023-10-28,fixed,RANGE_BASED,OPEN_PROXIMITY,3977.230965,-1243.07775,121,0.836789
2,2023-01-01 to 2023-10-28,2023-10-28 to 2023-12-27,fixed,RANGE_BASED,OPEN_PROXIMITY,2739.008368,-1699.99250,155,0.831716
3,2023-01-01 to 2023-12-27,2023-12-27 to 2024-02-25,fixed,RR_BASED,OPEN_PROXIMITY,4242.439547,-981.54885,157,0.902318
4,2023-01-01 to 2024-02-25,2024-02-25 to 2024-04-25,fixed,RR_BASED,OPEN_PROXIMITY,3438.908174,-4066.99875,197,0.694332
5,2023-01-01 to 2024-04-25,2024-04-25 to 2024-06-24,structural,RR_BASED,WORST_CASE,3180.877662,-134.77874,35,0.936048
6,2023-01-01 to 2024-06-24,2024-06-24 to 2024-08-23,structural,RR_BASED,WORST_CASE,3320.110390,-1532.03526,36,0.405765
7,2023-01-01 to 2024-08-23,2024-08-23 to 2024-10-22,fixed,RR_BASED,WORST_CASE,2487.409894,701.57230,89,1.124640
8,2023-01-01 to 2024-10-22,2024-10-22 to 2024-12-21,bos,RR_BASED,OPEN_PROXIMITY,3247.518551,-130.73458,91,0.976344


In [11]:
# --- D) Walk-forward: rolling (non-anchored) ---
wf_rolling = walk_forward_evaluation(
    df, swings, risk, configs,
    is_window_days=180,
    oos_window_days=60,
    anchored=False,
)

print("WF-ROLLING OOS METRICS:", wf_rolling["oos_metrics"])
display(wf_rolling["log"].head(20))


WF-ROLLING OOS METRICS: {'num_trades': 864, 'total_pnl': -2269.7610099998865, 'winrate': 0.34375, 'avg_r': -0.027300517090024932, 'profit_factor': 0.9544667810347807, 'max_dd_pnl': -3406.814009999886}


,train_period,test_period,best_sl,best_tp,best_same_bar,is_score,oos_total_pnl,oos_num_trades,oos_profit_factor
0,2023-01-01 to 2023-06-30,2023-06-30 to 2023-08-29,fixed,RANGE_BASED,OPEN_PROXIMITY,4740.388077,-616.99150,108,0.899746
1,2023-03-02 to 2023-08-29,2023-08-29 to 2023-10-28,fixed,RANGE_BASED,OPEN_PROXIMITY,3574.057291,-1243.07775,121,0.836789
2,2023-05-01 to 2023-10-28,2023-10-28 to 2023-12-27,structural,RR_BASED,OPEN_PROXIMITY,532.336488,-82.82210,45,0.954198
3,2023-06-30 to 2023-12-27,2023-12-27 to 2024-02-25,fixed,RR_BASED,WORST_CASE,1678.190385,164.78445,142,1.016722
4,2023-08-29 to 2024-02-25,2024-02-25 to 2024-04-25,bos,RR_BASED,WORST_CASE,2209.041157,-248.60270,103,0.961145
5,2023-10-28 to 2024-04-25,2024-04-25 to 2024-06-24,structural,RR_BASED,WORST_CASE,3021.381628,291.94000,40,1.122543
6,2023-12-27 to 2024-06-24,2024-06-24 to 2024-08-23,structural,RR_BASED,WORST_CASE,3194.995136,-796.44006,34,0.613071
7,2024-02-25 to 2024-08-23,2024-08-23 to 2024-10-22,fixed,RR_BASED,WORST_CASE,2500.389910,-35.65815,115,0.994069
8,2024-04-25 to 2024-10-22,2024-10-22 to 2024-12-21,fixed,RR_BASED,WORST_CASE,3521.088759,297.10680,156,1.039281


In [12]:
def _summarize_winner(
    time_split_result: Dict[str, Any],
    wf_anchored_result: Dict[str, Any],
    wf_rolling_result: Dict[str, Any],
) -> pd.DataFrame:
    """Create a compact comparison table."""
    rows = []

    rows.append(
        {
            "scheme": "time_split",
            "oos_total_pnl": float(time_split_result["test_metrics"]["total_pnl"]),
            "oos_num_trades": int(time_split_result["test_metrics"]["num_trades"]),
            "oos_profit_factor": float(time_split_result["test_metrics"]["profit_factor"]) if not pd.isna(time_split_result["test_metrics"]["profit_factor"]) else np.nan,
            "config": str(time_split_result["best_config"]),
        }
    )

    rows.append(
        {
            "scheme": "walk_forward_anchored",
            "oos_total_pnl": float(wf_anchored_result["oos_metrics"]["total_pnl"]),
            "oos_num_trades": int(wf_anchored_result["oos_metrics"]["num_trades"]),
            "oos_profit_factor": float(wf_anchored_result["oos_metrics"]["profit_factor"]) if not pd.isna(wf_anchored_result["oos_metrics"]["profit_factor"]) else np.nan,
            "config": "dynamic per window",
        }
    )

    rows.append(
        {
            "scheme": "walk_forward_rolling",
            "oos_total_pnl": float(wf_rolling_result["oos_metrics"]["total_pnl"]),
            "oos_num_trades": int(wf_rolling_result["oos_metrics"]["num_trades"]),
            "oos_profit_factor": float(wf_rolling_result["oos_metrics"]["profit_factor"]) if not pd.isna(wf_rolling_result["oos_metrics"]["profit_factor"]) else np.nan,
            "config": "dynamic per window",
        }
    )

    out = pd.DataFrame(rows)
    out = out.sort_values(["oos_total_pnl", "oos_profit_factor"], ascending=[False, False]).reset_index(drop=True)
    return out


comparison = _summarize_winner(time_split, wf_anchored, wf_rolling)
display(comparison)

winner = comparison.iloc[0]
print("\n=== WINNER (by OOS TotalPnL, then ProfitFactor) ===")
print(winner["scheme"], "|", "OOS PnL:", winner["oos_total_pnl"], "|", "PF:", winner["oos_profit_factor"])


,scheme,oos_total_pnl,oos_num_trades,oos_profit_factor,config
0,walk_forward_rolling,-2269.76101,864,0.954467,dynamic per window
1,time_split,-2806.05464,112,0.626512,"StrategyConfig(sl_mode='structural', tp_mode=<..."
2,walk_forward_anchored,-9704.58563,989,0.846123,dynamic per window



=== WINNER (by OOS TotalPnL, then ProfitFactor) ===
walk_forward_rolling | OOS PnL: -2269.7610099998865 | PF: 0.9544667810347807
